<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 6 · Giải tích cho ML</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Đạo hàm, quy tắc chuỗi, gradient và cách tự kiểm tra một công thức gradient: đủ giải tích để hiểu gradient descent và backpropagation.</p>
</div>

### Sau notebook này bạn sẽ

- Tính đạo hàm bằng sai phân và hiểu nó là độ dốc.
- Dùng quy tắc chuỗi, tính đạo hàm của sigmoid.
- Tính gradient của MSE theo w và b, khớp các con số của Chương 10.
- Bắt lỗi công thức gradient bằng gradient check.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch06-giai-tich/01-dao-ham/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Đạo hàm là độ dốc](#trang-1)
2. [Trang 2 · Quy tắc đạo hàm và quy tắc chuỗi](#trang-2)
3. [Trang 3 · Đạo hàm riêng và gradient](#trang-3)
4. [Trang 4 · Kiểm tra gradient bằng sai phân](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Đạo hàm là độ dốc

Từ cát tuyến tới tiếp tuyến, sai phân một phía và trung tâm.

*Đọc kèm trên web: [Trang 1 · Đạo hàm là độ dốc](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch06-giai-tich/01-dao-ham/)*

In [ ]:
import numpy as np                         # thư viện tính toán
f = lambda x: x ** 2                       # hàm f(x) = x bình phương
for h in [0.1, 0.01, 0.001]:               # thu ngắn dần đoạn h
    print(h, (f(3 + h) - f(3)) / h)        # sai phân một phía: tiến về 6
for h in [0.1, 0.01, 0.001]:               # cùng các h
    print(h, (f(3 + h) - f(3 - h)) / (2 * h))   # sai phân trung tâm: gần 6 ngay từ đầu

Output mong đợi:

```text
0.1 6.100000000000012
0.01 6.009999999999849
0.001 6.000999999999479
0.1 6.000000000000005
0.01 5.999999999999872
0.001 5.999999999999339
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Một phía: phần thừa đúng bằng h (6,1; 6,01; 6,001). Trung tâm: khử hết phần thừa, chỉ còn sai số dấu phẩy động.</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">h = 0,001 lại kém chính xác hơn h = 0,1 ở sai phân trung tâm: h quá nhỏ thì trừ hai số gần bằng nhau, mất chữ số có nghĩa.</p>
</div>

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Quy tắc đạo hàm và quy tắc chuỗi

Quy tắc chuỗi và hàm sigmoid.

*Đọc kèm trên web: [Trang 2 · Quy tắc đạo hàm và quy tắc chuỗi](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch06-giai-tich/02-quy-tac-chuoi/)*

In [ ]:
g = lambda x: (2 * x + 1) ** 2              # hàm lồng nhau
h = 1e-5                                    # bước sai phân
print((g(1 + h) - g(1 - h)) / (2 * h))      # quy tắc chuỗi cho 4(2x + 1) = 12 tại x = 1
sig = lambda z: 1 / (1 + np.exp(-z))        # hàm sigmoid
print(sig(0.0), sig(0.0) * (1 - sig(0.0)))  # giá trị và đạo hàm theo công thức (6.4)
print((sig(h) - sig(-h)) / (2 * h))         # kiểm tra bằng sai phân
print(round(sig(2.0), 4), round(sig(2.0) * (1 - sig(2.0)), 4))  # xa 0 thì đạo hàm nhỏ đi

Output mong đợi:

```text
12.000000000078613
0.5 0.25
0.2499999999933111
0.8808 0.105
```

<div style="border-left:6px solid #C0392B;background-color:#FBECEA;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#962D22;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">LỖI HAY GẶP</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Quên nhân đạo hàm lớp trong.</b> (2x + 1)² có đạo hàm 2(2x + 1) × 2, không phải 2(2x + 1).</p>
</div>

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Đạo hàm riêng và gradient

Gradient của MSE theo w và b.

*Đọc kèm trên web: [Trang 3 · Đạo hàm riêng và gradient](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch06-giai-tich/03-dao-ham-rieng-va-gradient/)*

In [ ]:
x = np.array([1, 2, 3, 4.0])                # giờ học của A, B, C, D
y = np.array([2, 4, 5, 7.0])                # điểm thi

def grad(w, b):                             # gradient của MSE theo (w, b)
    e = y - (w * x + b)                     # phần dư
    return float(-2 * np.mean(x * e)), float(-2 * np.mean(e))   # công thức (6.5)

print(grad(0, 0))          # điểm xuất phát của Chương 10
print(grad(1.6, 0.5))      # tại đáy: gần 0

Output mong đợi:

```text
(-26.5, -9.0)
(1.9984014443252818e-15, 6.661338147750939e-16)
```

<div style="border-left:6px solid #1F5F8B;background-color:#E9F1F8;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1F5F8B;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">TRỰC GIÁC</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Gradient chỉ hướng dốc lên. Tại (0, 0) cả hai thành phần âm, nên đi ngược hướng là tăng cả w lẫn b: đường y = 0 cần dốc hơn và nhấc lên.</p>
</div>

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Kiểm tra gradient bằng sai phân

So công thức với sai phân, bắt một lỗi quên hệ số 2.

*Đọc kèm trên web: [Trang 4 · Kiểm tra gradient bằng sai phân](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch06-giai-tich/04-kiem-tra-gradient/)*

In [ ]:
def L(w, b):                                          # MSE của bốn bạn
    return float(np.mean((y - (w * x + b)) ** 2))     # float để in gọn

def grad_so(w, b, h=1e-5):                            # gradient bằng sai phân trung tâm
    dw = (L(w + h, b) - L(w - h, b)) / (2 * h)        # nhích riêng w
    db = (L(w, b + h) - L(w, b - h)) / (2 * h)        # nhích riêng b
    return dw, db                                     # trả về cả hai

def grad_sai(w, b):                                   # công thức viết nhầm
    e = y - (w * x + b)                               # phần dư
    return float(-np.mean(x * e)), float(-2 * np.mean(e))   # QUÊN số 2 ở thành phần đầu

def lech(a, b):                                       # sai lệch tương đối
    return abs(a - b) / max(abs(a), abs(b))           # chia cho số lớn hơn

print(grad_so(0, 0))                                  # khớp (-26.5, -9.0)
so = grad_so(0.3, 0.2)                                # kiểm tra ở một điểm bất kỳ
print([round(lech(a, b), 8) for a, b in zip(grad(0.3, 0.2), so)])      # công thức đúng
print([round(lech(a, b), 8) for a, b in zip(grad_sai(0.3, 0.2), so)])  # công thức sai

Output mong đợi:

```text
(-26.499999999884945, -8.99999999965928)
[0.0, 0.0]
[0.5, 0.0]
```

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Đổi dấu trừ trong <code>grad_sai</code> thành dấu cộng ở thành phần b. Sai lệch thành bao nhiêu?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2.</b> Chạy kiểm tra tại (1.6, 0.5). Vì sao kết quả ở đáy khó tin hơn?</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Tính tay trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch06-giai-tich/05-bai-tap/)*

### Bài 6.1 · Chỗ mặt đất bằng phẳng

Cho $k(x) = x^3 - 3x$. Tính $k'(2)$, tìm mọi $x$ có $k'(x) = 0$.
<details><summary>Xem đáp án</summary>

$k'(x) = 3x^2 - 3$, $k'(2) = 9$. $k'(x) = 0$ tại $x = \pm 1$: một đỉnh đồi và một thung lũng nhỏ, không phải đáy thấp nhất. Code kiểm tra ở ô ngay dưới.
</details>

In [ ]:
k = lambda x: x ** 3 - 3 * x              # hàm bậc ba
print((k(2 + h) - k(2 - h)) / (2 * h))    # phải gần 9

Output mong đợi:

```text
9.000000000147779
```

### Bài 6.2 · Chuỗi qua sigmoid

Với $p = \sigma(2w)$ và loss $(p - 1)^2$, tính đạo hàm theo $w$ tại $w = 0$.
<details><summary>Xem đáp án</summary>

Ba lớp: $2(p - 1) = -1$, $\sigma(1 - \sigma) = 0{,}25$, $dz/dw = 2$. Nhân lại $-0{,}5$. Code kiểm tra ở ô ngay dưới.
</details>

In [ ]:
q = lambda w: (sig(w * 2) - 1) ** 2       # loss theo w, với x = 2 và nhãn 1
print((q(h) - q(-h)) / (2 * h))           # phải gần -0.5

Output mong đợi:

```text
-0.4999999999810711
```

### Bài 6.3 · Bước hai của Chương 10

Từ $w = 1{,}325$, $b = 0{,}45$, tính gradient và $w$, $b$ sau một bước với $\eta = 0{,}05$.
<details><summary>Xem đáp án</summary>

$\sum x_i e_i = 8{,}75$, $\sum e_i = 2{,}95$. Gradient $(-4{,}375;\ -1{,}475)$. Sau bước: $w = 1{,}54375$, $b = 0{,}52375$. Code kiểm tra ở ô ngay dưới.
</details>

In [ ]:
gw, gb = grad(1.325, 0.45)                 # gradient tại sau bước 1
print(gw, gb)                              # hai thành phần
print(1.325 - 0.05 * gw, 0.45 - 0.05 * gb) # bước cập nhật thứ hai

Output mong đợi:

```text
-4.375 -1.475
1.54375 0.52375
```